<a href="https://colab.research.google.com/github/Angappan95/async-practice/blob/main/01_asyncio_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install nest_asyncio httpx uvicorn -q

In [3]:
import asyncio
import time

async def fetch(name, delay=1):
    print(f"Initiated - {name}")
    await asyncio.sleep(delay)
    return name

# Async with task
async def fetch_with_task():
    task = asyncio.create_task(fetch("Background", 2))
    await asyncio.sleep(1)
    result = await task
    print(result)

await fetch_with_task()

Initiated - Background
Background


In [5]:
# Async with taskgroup
import asyncio

async def fetch_data():
    await asyncio.sleep(2)
    return "Data fetched"

async def fetch_with_taskgroup():
    async with asyncio.TaskGroup() as group:
        task1 = group.create_task(fetch_data())
        task2 = group.create_task(fetch_data())

    print(task1.result())
    print(task2.result())

await fetch_with_taskgroup()

Data fetched
Data fetched


In [7]:
import httpx
from fastapi import FastAPI
import asyncio
import time

app = FastAPI()

@app.get("/provider/{provider_id}")
async def provider(provider_id: int):
    await asyncio.sleep(2)

    return {
        "doc_id": f"doc-{provider_id}",
        "score": provider_id * 0.25
    }

@app.get("/aggregate")
async def aggregate():
    async with httpx.AsyncClient() as client:
        responses = await asyncio.gather(*[
            client.get(f"http://127.0.0.1:8000/provider/{i}")
            for i in range(1, 4)
        ])

    for response in responses:
        response.raise_for_status()

    documents = [response.json() for response in responses]

    return {
        "best_document": max(
            documents,
            key=lambda doc: doc["score"]
        ),
        "documents": documents
    }

In [8]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start the server in a separate background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

In [9]:
import httpx
import asyncio

async def test_api():
    async with httpx.AsyncClient() as client:
        response = await client.get("http://127.0.0.1:8000/aggregate")
        print("Status Code:", response.status_code)
        print("JSON Response:")
        import json
        print(json.dumps(response.json(), indent=2))

# Run the client request
await test_api()

INFO:     127.0.0.1:48028 - "GET /provider/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:48042 - "GET /provider/2 HTTP/1.1" 200 OK
INFO:     127.0.0.1:48044 - "GET /provider/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:48012 - "GET /aggregate HTTP/1.1" 200 OK
Status Code: 200
JSON Response:
{
  "best_document": {
    "doc_id": "doc-3",
    "score": 0.75
  },
  "documents": [
    {
      "doc_id": "doc-1",
      "score": 0.25
    },
    {
      "doc_id": "doc-2",
      "score": 0.5
    },
    {
      "doc_id": "doc-3",
      "score": 0.75
    }
  ]
}


In [12]:
# Semaphore
import httpx
import asyncio

semaphore = asyncio.Semaphore(2)

async def limited_fetch(client, provider_id):
    async with semaphore:
        response = await client.get(f"http://127.0.0.1:8000/provider/{provider_id}", timeout=2.5)
        response.raise_for_status()
        return response.json()

async def run_limited_fetches():
    async with httpx.AsyncClient() as client:
        results = await asyncio.gather(*[limited_fetch(client, i) for i in range(1, 10)])
        print(f"Successfully fetched {len(results)} items.")
        return results

await run_limited_fetches()

INFO:     127.0.0.1:52162 - "GET /provider/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52166 - "GET /provider/2 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52162 - "GET /provider/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52166 - "GET /provider/4 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52162 - "GET /provider/5 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52166 - "GET /provider/6 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52162 - "GET /provider/7 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52166 - "GET /provider/8 HTTP/1.1" 200 OK
INFO:     127.0.0.1:52162 - "GET /provider/9 HTTP/1.1" 200 OK
Successfully fetched 9 items.


[{'doc_id': 'doc-1', 'score': 0.25},
 {'doc_id': 'doc-2', 'score': 0.5},
 {'doc_id': 'doc-3', 'score': 0.75},
 {'doc_id': 'doc-4', 'score': 1.0},
 {'doc_id': 'doc-5', 'score': 1.25},
 {'doc_id': 'doc-6', 'score': 1.5},
 {'doc_id': 'doc-7', 'score': 1.75},
 {'doc_id': 'doc-8', 'score': 2.0},
 {'doc_id': 'doc-9', 'score': 2.25}]